# 02 — VH Backscatter Threshold Baseline

Tunes a single VH dB threshold (`VH < threshold => water`) strictly on the **validation split**, then evaluates water-class IoU, F1, precision, and recall on:
- `val` (89 chips)
- `test` (90 chips)
- `bolivia` (15 chips, held-out generalization event)

In [ ]:
import json, sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from ml import config
from ml.src.dataset import read_split_csv
from ml.src.evaluate import tune_and_evaluate_baseline

val_pairs = read_split_csv(config.SPLITS_DIR / 'flood_valid_data.csv')
test_pairs = read_split_csv(config.SPLITS_DIR / 'flood_test_data.csv')
bolivia_pairs = read_split_csv(config.SPLITS_DIR / 'flood_bolivia_data.csv')

baseline = tune_and_evaluate_baseline(val_pairs, test_pairs, bolivia_pairs)
print('Best VH threshold (dB, tuned on val):', baseline['best_vh_threshold_db'])
for split_name in ('val', 'test', 'bolivia'):
    m = baseline[split_name]
    print(f"{split_name:8s} | IoU={m['iou']:.4f} | F1={m['f1']:.4f} | P={m['precision']:.4f} | R={m['recall']:.4f}")

config.CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
(config.CHECKPOINTS_DIR / 'baseline_metrics.json').write_text(json.dumps(baseline, indent=2))
print('Saved baseline_metrics.json')